In [1]:
from crewai import Agent, Task, Crew
from crewai_tools import EXASearchTool, ScrapeWebsiteTool
import os
from utils import get_openai_api_key, get_exa_api_key, load_env
from IPython.display import Markdown
import yaml

# Load .env first — sets OPENAI_API_BASE, OPENAI_BASE_URL, OPENAI_MODEL_NAME
load_env()

# set up the OpenAI API key
os.environ["OPENAI_API_KEY"] = get_openai_api_key()
# set the EXA API key
os.environ["EXA_API_KEY"] = get_exa_api_key()


Using Custom API Base: https://api.deepseek.com
Using Model: deepseek-v4-flash
Using Custom API Base: https://api.deepseek.com
Using Model: deepseek-v4-flash
Using Custom API Base: https://api.deepseek.com
Using Model: deepseek-v4-flash


In [ ]:
# create the tool instances
exa_search_tool = EXASearchTool(base_url=os.getenv("EXA_BASE_URL")) 
print("... ",os.getenv("EXA_BASE_URL"))
scrape_website_tool = ScrapeWebsiteTool()

# load the configuration file for the agents
with open('config/agents.yaml', 'r') as file:
        agent_config = yaml.safe_load(file)

# create the agents using the configuration
research_planner = Agent(
        config=agent_config['research_planner'],
        verbose=True,
        max_rpm=30, # 150
        max_iter=15  # 15
        )
internet_researcher = Agent(
        config=agent_config['internet_researcher'],
        tools=[exa_search_tool, scrape_website_tool],
        verbose=True,
        max_rpm=30, # 150
        max_iter=15  # 15
        )
fact_checker = Agent(
        config=agent_config['fact_checker'],
        tools=[exa_search_tool, scrape_website_tool],
        verbose=True,
        max_rpm=30, # 150
        max_iter=15  # 15
        )
report_writer = Agent(
        config=agent_config['report_writer'],
        verbose=True,
        max_rpm=30, # 150
        max_iter=15  # 15
        )

...  None


In [3]:
import re

# write the custom guardrail function
def write_report_guardrail(output):
    # get the raw output from the TaskOutput object
    try:
        output = output if type(output)==str else output.raw 
    except Exception as e:
        return (False, ("Error retrieving the `raw` argument: "
                        f"\n{str(e)}\n"
                        )
                )
    
    # convert the output to lowercase
    output_lower = output.lower()

    # check that the summary section exists
    if not re.search(r'#+.*summary', output_lower):
        return (False, 
                "The report must include a Summary section with a header like '## Summary'"
                )

    # check that the insights or recommendations sections exist
    if not re.search(r'#+.*insights|#+.*recommendations', output_lower):
        return (False, 
                "The report must include an Insights section with a header like '## Insights'"
                )

    ### START CODE HERE ###

    # check that the citations (or references) section exists
    if not re.search(r'#+.*citations|#+.*references', output_lower):
        return (False,
                "The report must include a Citations (or References) section with a header like '## Citations'"
                )

    ### END CODE HERE ###
    return (True, output)


In [4]:
test_report_pass = """
# Report title

## Executive Summary
This is a summary.

## Insights
These are the insights.

## Citations
1. Citation 1
2. Citation 2
"""

write_report_guardrail(test_report_pass)

(True,
 '\n# Report title\n\n## Executive Summary\nThis is a summary.\n\n## Insights\nThese are the insights.\n\n## Citations\n1. Citation 1\n2. Citation 2\n')

In [5]:
test_report_fail = """
# Report title

## Executive Summary
This is a summary.
"""

write_report_guardrail(test_report_fail)

(False,
 "The report must include an Insights section with a header like '## Insights'")

In [6]:
# load the configuration file for the tasks
with open('config/tasks.yaml', 'r') as file:
    task_config = yaml.safe_load(file)

### START CODE HERE ###

# create the tasks using the configuration
create_research_plan = Task(
    config=task_config['create_research_plan'],
    agent=research_planner
)

gather_research_data = Task(
    config=task_config['gather_research_data'],
    agent=internet_researcher,
)

verify_information_quality = Task(
    config=task_config['verify_information_quality'],
    agent=fact_checker,
)

write_final_report = Task(
    config=task_config['write_final_report'],
    agent=report_writer,
    guardrails=[write_report_guardrail],
)

### END CODE HERE ###

In [7]:
def save_file_hook(result):
    """
    Save the final research report to a local markdown file
    """
    try:
        # Get the final report content from the last task output
        if hasattr(result, 'tasks_output') and result.tasks_output:
            report_content = result.tasks_output[-1].raw
        else:
            report_content = str(result)
        
        filename = f"research_report.md"
        
        # Save to file
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(report_content)
        
        print(f"Report successfully saved to: {filename}")
        
    except Exception as e:
        print(f"Error saving report to file: {str(e)}")

In [8]:
# Create the urban planning crew
deep_research_crew = Crew(
    # include all the agents
    agents=[research_planner, 
            internet_researcher, 
            fact_checker, 
            report_writer],
    # include all the tasks in the order to be executed
    tasks=[create_research_plan, 
           gather_research_data, 
           verify_information_quality, 
           write_final_report],

    ### START CODE HERE ###
    
    # add memory to the crew
    # memory=True,
    memory=False, # set to True to enable memory across tasks, False to disable memory
    # add the after kickoff hook
    after_kickoff_callbacks=[save_file_hook]

    ### END CODE HERE ###
)

In [9]:
### START CODE HERE ###

# write your query in the "user_query" value
inputs = {
    "user_query": "Evaluate the top one emerging AI tool for automating competitive market analysis, including its features, limitations, costs, and ideal use cases for a mid-sized marketing firm."
}

### END CODE HERE ###

In [10]:
# Execute the crew's tasks
result = deep_research_crew.kickoff(inputs=inputs)

/Users/taozh/Documents/agents/agent-jupyter/venv/lib/python3.13/site-packages/pydantic/main.py:250: UserWarning: method callbacks cannot be serialized and will prevent checkpointing. Use a module-level named function instead.
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Planner                                                                                        │
│                                                                                                                 │
│  Task: Break down the research query "Evaluate the top one emerging AI tool for automating competitive market   │
│  analysis, including its features, limitations, costs, and ideal use cases for a mid-sized marketing firm."     │
│  into specific topics and key questions that need investigation. Create a focused research plan.                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Planner                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## Research Plan: Evaluating the Top Emerging AI Tool for Automating Competitive Market Analysis               │
│                                                                                                                 │
│  This research plan breaks down the original query into specific topics, key questions, and success criteria    │
│  to systematically identify and evaluate the leading emerging AI tool for a mid-sized marketing firm.           │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1. Main Research Topics to Investigate                                                                     │
│                                                                                                                 │
│  - **Topic A: Identification and Selection of the Top Emerging AI Tool**                                        │
│    *Focus: Determine which tool is currently considered the leading emerging solution for automated             │
│  competitive market analysis.*                                                                                  │
│                                                                                                                 │
│  - **Topic B: Features of the Tool**                                                                            │
│    *Focus: Detailed capabilities, AI functions, integrations, and user interface.*                              │
│                                                                                                                 │
│  - **Topic C: Limitations of the Tool**                                                                         │
│    *Focus: Known drawbacks, data quality issues, learning curve, and gaps in functionality.*                    │
│                                                                                                                 │
│  - **Topic D: Costs and Pricing Model**                                                                         │
│    *Focus: Subscription tiers, total cost of ownership, hidden fees, and value for a mid-sized firm.*           │
│                                                                                                                 │
│  - **Topic E: Ideal Use Cases for a Mid-Sized Marketing Firm**                                                  │
│    *Focus: Practical applications, ROI scenarios, workflow fit, and company size considerations.*               │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 2. Key Questions for Each Topic                                                                            │
│                                                                                                                 │
│  #### Topic A – Identification and Selection           

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Internet Researcher                                                                                     │
│                                                                                                                 │
│  Task: Using the research plan, search the internet and scrape relevant websites to collect comprehensive       │
│  information on all identified topics. Verify information across multiple sources and cite all sources used.    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool exa_search_tool executed with result: Title: IntelCue | AI Competitive Intelligence Platform & Market Monitoring
URL: https://www.intelcue.ai/
ID: https://www.intelcue.ai/
Score: None
Published Date: 2024-11-15T00:00:00.000Z
Author: None
...
Tool exa_search_tool executed with result: Title: 6 Competitive Intelligence Tools to Consider for 2024
URL: https://www.palvdm.com/blog/best-competitive-intelligence-tools
ID: https://www.palvdm.com/blog/best-competitive-intelligence-tools
Sc...
Tool exa_search_tool executed with result: Title: Market Guide for Competitive and Market Intelligence Tools
URL: https://www.gartner.com/en/documents/5487795
ID: https://www.gartner.com/en/documents/5487795
Score: None
Published Date: 2024-06...
Tool exa_search_tool executed with result: Title: Crayon Competitive Intelligence Platform: Complete Buyer's Guide
URL: https://www.staymodern.ai/solutions/crayon-competitive-intelligence-platform
ID: https://www.staymodern.ai/solutions/crayon...
Tool exa_sea

KeyboardInterrupt: 